In [1]:
import pandas as pd
import os
import glob
import tkinter as tk
from tkinter import filedialog

In [2]:
# ダイアログの裏回りを防ぐための設定
root = tk.Tk()
root.withdraw() # 余分なメインウィンドウを隠す
root.attributes('-topmost', True) # ダイアログを常に最前面に表示する

# フォルダ選択ダイアログを表示
print("フォルダ選択ダイアログを開いています...")
target_dir = filedialog.askdirectory(title="CSVファイルが入っているメインフォルダを選択してください（サブフォルダ内も対象）")

if not target_dir:
    print("キャンセルされました。対象フォルダが選択されていません。")
else:
    print(f"選択されたフォルダ:\n{target_dir}\n")
    print("次のセルを実行すると、このフォルダとサブフォルダ内のCSVファイルを処理します。")

フォルダ選択ダイアログを開いています...
選択されたフォルダ:
C:/Users/0uh2j/Desktop/vscodeで/09_10行ごとの平均と標準偏差上下算出と出力(主に温度)/データ

次のセルを実行すると、このフォルダとサブフォルダ内のCSVファイルを処理します。


In [3]:
# 条件を表す列（2〜4列目）。この3列の値の組み合わせが同じ行を「同じ条件」として集計する
COND_COLS = ['キャビティ厚み[mm]', '加熱シリンダ温度[℃]', '射出速度[mm/s]']

# 出力ファイル名の末尾。過去の出力（旧名を含む）は処理対象から外す
OUTPUT_SUFFIX = '_条件ごと集計結果'
SKIP_MARKERS = [OUTPUT_SUFFIX, '_10行ごと集計結果']

# 読み込みを試す文字コードの順番（UTF-8で読めなければ、Excelで保存したShift-JISのCSVとして読む）
ENCODINGS = ['utf-8', 'cp932']


def read_csv_any_encoding(path, **kwargs):
    """ENCODINGS の順に文字コードを試してCSVを読み込み、(データフレーム, 使った文字コード) を返す"""
    for enc in ENCODINGS:
        try:
            return pd.read_csv(path, encoding=enc, **kwargs), enc
        except UnicodeDecodeError:
            continue
    raise ValueError(f"文字コード {ENCODINGS} のどれでも読み込めませんでした")


def find_csv_files(main_dir):
    """メインフォルダと、その下の末端フォルダまでのすべてのフォルダからCSVファイルを集める"""
    found = []
    for dirpath, dirnames, filenames in os.walk(main_dir):
        dirnames.sort()  # フォルダ名の順に処理する
        for name in sorted(filenames):
            if not name.lower().endswith('.csv'):
                continue
            if name.startswith('~$'):  # Excelでファイルを開いているときにできる一時ファイル
                continue
            if any(m in name for m in SKIP_MARKERS):  # 出力済みの集計結果ファイル
                continue
            found.append(os.path.join(dirpath, name))
    return found


if not target_dir:
    print("セル2でフォルダが選択されていません。もう一度セル2を実行してください。")
else:
    # 選択したメインフォルダ以下のすべてのフォルダからCSVファイルを集める
    candidate_files = find_csv_files(target_dir)

    if not candidate_files:
        print("指定されたフォルダ内に処理対象のCSVファイルが見つかりませんでした。")
    else:
        print(f"CSVファイルが {len(candidate_files)} 件見つかりました。\n" + "-"*30)
        done_files, skipped_files, error_files = [], [], []

        for file_path in candidate_files:
            rel_path = os.path.relpath(file_path, target_dir)
            print(f"処理中: {rel_path} ...")

            try:
                # 列名だけ先に読み、2〜4列目が条件の列になっているファイルだけを処理する
                header, _ = read_csv_any_encoding(file_path, nrows=0)
                if header.columns[1:4].tolist() != COND_COLS:
                    print(f"  - スキップ: 2〜4列目が {COND_COLS} ではないため、加工対象外と判断しました\n")
                    skipped_files.append(rel_path)
                    continue

                # CSVファイルの読み込み
                df, enc = read_csv_any_encoding(file_path)
                if enc != 'utf-8':
                    print(f"  ※ 文字コード {enc} として読み込みました")

                # 最初の4列（各条件の1行目の値をそのまま使う列）と、それ以降の列（計算する列）を分ける
                first_4_cols = df.columns[:4].tolist()
                calc_cols = df.columns[4:].tolist()

                # 計算する列に数値以外（文字や「#DIV/0!」など）が混ざっていたら、その値は空欄として扱う
                non_numeric_cols = [c for c in calc_cols if not pd.api.types.is_numeric_dtype(df[c])]
                if non_numeric_cols:
                    n_blank_before = int(df[non_numeric_cols].isna().sum().sum())
                    df[non_numeric_cols] = df[non_numeric_cols].apply(pd.to_numeric, errors='coerce')
                    n_converted = int(df[non_numeric_cols].isna().sum().sum()) - n_blank_before
                    print(f"  ※ 数値でない値が {n_converted} 個あったため、空欄として扱いました（列: {non_numeric_cols}）")

                # 条件（2〜4列目の組み合わせ）ごとにグループ番号を振る
                # ファイル内で先に出てきた条件から順に 0, 1, 2, ... となる
                group_ids = df.groupby(COND_COLS, sort=False, dropna=False).ngroup()

                # 条件ごとの行数を確認（外れ値除外などで条件ごとに行数が異なっていてもよい）
                group_sizes = group_ids.value_counts()
                print(f"  条件数: {len(group_sizes)} / 1条件あたりの行数: {group_sizes.min()}〜{group_sizes.max()}")
                n_single = int((group_sizes == 1).sum())
                if n_single > 0:
                    print(f"  ※ 1行しかない条件が {n_single} 件あります（標準偏差は0として出力します）")

                # 最初の4列は、各条件の最初の行の値を取得
                df_first = df[first_4_cols].groupby(group_ids).first()

                # 残りの列（5列目以降）は、条件ごとに平均と標準偏差を計算
                df_calc = df[calc_cols]
                df_mean = df_calc.groupby(group_ids).mean()
                df_std = df_calc.groupby(group_ids).std().fillna(0)

                # 新しい列名リストを作成
                new_cols = first_4_cols.copy()
                for col in calc_cols:
                    new_cols.extend([f'{col}_平均', f'{col}_標準偏差上', f'{col}_標準偏差下'])

                # 結果を格納するデータフレームを作成し、値を代入
                df_result = pd.DataFrame(index=df_mean.index, columns=new_cols)

                for col in first_4_cols:
                    df_result[col] = df_first[col]

                for col in calc_cols:
                    mean_series = df_mean[col]
                    std_series = df_std[col]
                    df_result[f'{col}_平均'] = mean_series
                    df_result[f'{col}_標準偏差上'] = mean_series + std_series
                    df_result[f'{col}_標準偏差下'] = mean_series - std_series

                # 出力ファイルパスの作成（元のCSVと同じフォルダに保存）
                output_filename = os.path.splitext(os.path.basename(file_path))[0] + OUTPUT_SUFFIX + '.csv'
                output_path = os.path.join(os.path.dirname(file_path), output_filename)

                # CSV書き出し
                df_result.to_csv(output_path, encoding='utf-8-sig', index=False)
                print(f"  → 出力完了: {output_filename}\n")
                done_files.append(rel_path)

            except Exception as e:
                # 何らかのエラーが起きた場合はスキップして次のファイルへ
                print(f"  × エラーが発生しました: {rel_path} (理由: {e})\n")
                error_files.append(rel_path)

        print("-" * 30)
        print(f"すべてのファイルの処理が完了しました！ 出力 {len(done_files)} 件 / スキップ {len(skipped_files)} 件 / エラー {len(error_files)} 件")
        if error_files:
            print("\nエラーで出力できなかったファイル（Excelで開いたままになっていないかなどを確認してください）:")
            for p in error_files:
                print(f"  {p}")

CSVファイルが 3 件見つかりました。
------------------------------
処理中: 01_pp_抽出後換算結果\04_pp_all_1.5-3.0なし\最高値外れ値行除外_pp_all(1.5_3.0なし)_複合換算結果_summary_all_features.csv ...
  条件数: 126 / 1条件あたりの行数: 8〜10
  → 出力完了: 最高値外れ値行除外_pp_all(1.5_3.0なし)_複合換算結果_summary_all_features_条件ごと集計結果.csv

処理中: 02_mixpp_抽出後換算結果\01_mixpp_Alldata\最高値外れ値行除外_mixpp_Alldata_relative-value_max_summary_all_features.csv ...
  条件数: 129 / 1条件あたりの行数: 2〜5
  → 出力完了: 最高値外れ値行除外_mixpp_Alldata_relative-value_max_summary_all_features_条件ごと集計結果.csv

処理中: 03_gpps_抽出後換算結果\gpps_alldata_20261014\最高値外れ値行除外_換算結果_gpps_NRalldata_max_summary_all_features.csv ...
  条件数: 102 / 1条件あたりの行数: 2〜5
  → 出力完了: 最高値外れ値行除外_換算結果_gpps_NRalldata_max_summary_all_features_条件ごと集計結果.csv

------------------------------
すべてのファイルの処理が完了しました！ 出力 3 件 / スキップ 0 件 / エラー 0 件
